<a href="https://www.kaggle.com/code/avikdas567/imagine-decoding-challenge?scriptVersionId=307727096" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import os
import numpy as np
import pandas as pd
import mne
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.pipeline import Pipeline
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# PATHS

TRAIN_PATH = "/kaggle/input/competitions/the-imagine-decoding-challenge/train/train"
TEST_PATH = "/kaggle/input/competitions/the-imagine-decoding-challenge/test/test"

# CONFIG

LOCALIZER_WINDOWS = [(0.05,0.2),(0.1,0.3),(0.2,0.4)]
IMAGINE_WINDOWS = [(0,1),(1,2),(2,3),(3,4)]
N_PCA = 128
BATCH_SIZE = 64
EPOCHS = 6

# HELPERS

def load_epochs(path):
    epochs = mne.read_epochs(path, preload=True, verbose=False)
    picks = mne.pick_types(epochs.info, meg=True, eeg=False, eog=False, ecg=False)
    epochs.pick(picks)
    return epochs

def extract_features(data, times, window):
    tmin, tmax = window
    idx = np.where((times >= tmin) & (times <= tmax))[0]
    chunk = data[:, :, idx]
    
    mean_feat = chunk.mean(axis=2)
    std_feat = chunk.std(axis=2)
    max_feat = chunk.max(axis=2)
    
    return np.concatenate([mean_feat, std_feat, max_feat], axis=1)

def get_label_mapping(event_id):
    labels = sorted(event_id.keys())
    mapping = {event_id[k]: k for k in labels}
    return mapping, labels

# LABEL MAPPING

sample_sub = sorted(os.listdir(TRAIN_PATH))[0]
sample_epochs = load_epochs(os.path.join(TRAIN_PATH, sample_sub, f"{sample_sub}_localizer-epo.fif"))
mapping, label_names = get_label_mapping(sample_epochs.event_id)

# TRAIN CLASSICAL MODELS

models = []

print("Training classical models...")

for window in LOCALIZER_WINDOWS:
    Xw, yw = [], []
    
    for sub in tqdm(sorted(os.listdir(TRAIN_PATH)), leave=False):
        sub_path = os.path.join(TRAIN_PATH, sub)
        loc_file = os.path.join(sub_path, f"{sub}_localizer-epo.fif")
        
        if not os.path.exists(loc_file):
            continue
        
        epochs = load_epochs(loc_file)
        data = epochs.get_data()
        times = epochs.times
        
        data = (data - data.mean()) / (data.std() + 1e-6)
        
        feats = extract_features(data, times, window)
        y = epochs.events[:, 2]
        
        Xw.append(feats)
        yw.append(y)
    
    Xw = np.vstack(Xw)
    yw = np.array([mapping[i] for i in np.concatenate(yw)])
    
    ridge = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=N_PCA)),
        ("clf", RidgeClassifier())
    ])
    
    logreg = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=N_PCA)),
        ("clf", LogisticRegression(max_iter=300, n_jobs=-1))
    ])
    
    ridge.fit(Xw, yw)
    logreg.fit(Xw, yw)
    
    models.append((window, ridge, logreg))

print("Classical models trained.")

# CNN MODEL

class SimpleCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, (1,15), padding=(0,7))
        self.conv2 = nn.Conv2d(16, 32, (306,1))
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(32, n_classes)
    
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

print("Training CNN...")

X_dl = []
y_dl = []

for sub in tqdm(sorted(os.listdir(TRAIN_PATH))):
    sub_path = os.path.join(TRAIN_PATH, sub)
    loc_file = os.path.join(sub_path, f"{sub}_localizer-epo.fif")
    
    if not os.path.exists(loc_file):
        continue
    
    epochs = load_epochs(loc_file)
    data = epochs.get_data()
    data = (data - data.mean()) / (data.std() + 1e-6)
    
    X_dl.append(data)
    y_dl.append(epochs.events[:,2])

X_dl = torch.tensor(np.vstack(X_dl)).float().unsqueeze(1)
y_dl = torch.tensor([label_names.index(mapping[i]) for i in np.concatenate(y_dl)]).long()

dataset = TensorDataset(X_dl, y_dl)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = SimpleCNN(len(label_names)).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for xb, yb in tqdm(loader, leave=False, desc=f"Epoch {epoch+1}"):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        
        optimizer.zero_grad()
        out = model(xb)
        loss = loss_fn(out, yb)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss:.4f}")

print("CNN model trained.")

# PREDICTION

def predict_subject(sub_path):
    imag_file = os.path.join(sub_path, f"{os.path.basename(sub_path)}_imagine-epo.fif")
    
    epochs = load_epochs(imag_file)
    data = epochs.get_data()
    times = epochs.times
    
    data = (data - data.mean()) / (data.std() + 1e-6)
    
    all_scores = []
    
    for (window, ridge, logreg) in models:
        feats = extract_features(data, times, window)
        r = ridge.decision_function(feats)
        l = logreg.predict_proba(feats)
        all_scores.append(r + l)
    
    classical_scores = np.mean(np.stack(all_scores), axis=0)
    
    X_t = torch.tensor(data).float().unsqueeze(1).to(DEVICE)
    model.eval()
    with torch.no_grad():
        dl_scores = model(X_t).cpu().numpy()
    
    final_scores = classical_scores + 0.2 * dl_scores
    final_idx = final_scores.argmax(axis=1)
    
    return [label_names[i] for i in final_idx]

# TEST

print("Predicting TEST...")

submission = []

for sub in tqdm(sorted(os.listdir(TEST_PATH))):
    sub_path = os.path.join(TEST_PATH, sub)
    
    preds = predict_subject(sub_path)
    
    for i, p in enumerate(preds):
        submission.append({
            "ID": f"{sub}_{i+1}",
            "label": p
        })

df = pd.DataFrame(submission)
df.to_csv("submission.csv", index=False)

print("\n'submission.csv' created.")
display(df.head())

Training classical models...


Classical models trained.
Training CNN...


100%|██████████| 15/15 [00:09<00:00,  1.64it/s]


Epoch 1/6 - Loss: 260.6111


Epoch 2/6 - Loss: 260.3055


...

Epoch 5/6 - Loss: 260.3024


Epoch 6/6 - Loss: 260.3186
CNN model trained.
Predicting TEST...


100%|██████████| 14/14 [00:09<00:00,  1.44it/s]


'submission.csv' created.


,ID,label
0,sub-01_1,mountain
1,sub-01_2,foot
2,sub-01_3,clown
3,sub-01_4,clown
4,sub-01_5,clown
